In [1]:
import sys
import os

print("Python Version:", sys.version)
print("Project Location:", os.getcwd())
print("Python Executable:", sys.executable)

Python Version: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
Project Location: C:\Users\Sonu Satra\Downloads\Downloads\sem-05\MLops SCSE3040\G136_MP026
Python Executable: C:\Users\Sonu Satra\Downloads\Downloads\sem-05\MLops SCSE3040\G136_MP026\.venv\Scripts\python.exe


In [3]:
%pip install pandas numpy scikit-learn mlflow joblib pyyaml

  Using cached pandas-3.1.1-cp314-cp314-win_amd64.whl.metadata (16 kB)
  Using cached numpy-2.5.5-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached mlflow-3.18.0-py3-none-any.whl.metadata (50 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached narwhals-2.29.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.8.0-py3-none-any.whl.metadata (29 kB)
  Using cached mlflow_skinny-3.18.0-py3-none-any.whl.metadata (51 kB)
  Using cached mlflow_tracing-3.18.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.5-py3-none-any.whl.metadata (5.4 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.20.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached cryptography-50.0.2-cp311-abi3-win_amd64.whl.metadata (4.4 kB)
  Using cached docker-7.2.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import sklearn
import mlflow
import joblib

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("MLflow:", mlflow.__version__)
print("Joblib:", joblib.__version__)

print("\nAll libraries working!")

Pandas: 3.1.1
NumPy: 2.5.5
Scikit-learn: 1.9.1
MLflow: 3.18.0
Joblib: 1.6.0

All libraries working!


In [2]:
import os

for root, dirs, files in os.walk("data"):
    dirs[:] = [d for d in dirs if d != ".dvc"]
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

data\processed\train_clean.csv
data\raw\holdout_features.csv
data\raw\later.csv
data\raw\train.csv


Member 1 Dataset Verification

In [3]:

import pandas as pd

raw = pd.read_csv("data/raw/train.csv")
clean = pd.read_csv("data/processed/train_clean.csv")

print("Original dataset:", raw.shape)
print("Processed dataset:", clean.shape)

print("\nOriginal duplicates:", raw.duplicated().sum())
print("Processed duplicates:", clean.duplicated().sum())

print("\nProcessed columns:")
print(clean.columns.tolist())

Original dataset: (1039, 12)
Processed dataset: (1000, 11)

Original duplicates: 39
Processed duplicates: 0

Processed columns:
['record_id', 'date', 'loom_age_yr', 'thread_count', 'power_cuts_min', 'weaver_exp_yr', 'humidity_pct', 'shift_hours', 'fabric', 'shed', 'metres_per_shift']


In [4]:
import pandas as pd

raw = pd.read_csv("data/raw/train.csv")
clean = pd.read_csv("data/processed/train_clean.csv")

# 1. Duplicate check
print("Original duplicates:", raw.duplicated().sum())
print("Clean duplicates:", clean.duplicated().sum())

# 2. Leakage column check
print("\nLeakage in raw:",
      "metres_checked_at_folding" in raw.columns)

print("Leakage in clean:",
      "metres_checked_at_folding" in clean.columns)

# 3. Row count check
print("\nOriginal rows:", len(raw))
print("Clean rows:", len(clean))

# 4. Target column check
print("\nTarget available:",
      "metres_per_shift" in clean.columns)

# 5. Stuck sensor check
print("\nStuck sensor rows in raw:")
print(
    raw[
        (raw["date"] >= "2026-03-01") &
        (raw["date"] <= "2026-03-20") &
        (raw["shift_hours"] == 10.8)
    ].shape[0]
)

Original duplicates: 39
Clean duplicates: 0

Leakage in raw: True
Leakage in clean: False

Original rows: 1039
Clean rows: 1000

Target available: True

Stuck sensor rows in raw:
101


In [5]:
raw_shed1 = raw[raw["shed"] == "Shed 1"]

clean_shed1 = clean[clean["shed"] == "Shed 1"]

print("Original Shed 1 mean:",
      raw_shed1["power_cuts_min"].mean())

print("Clean Shed 1 mean:",
      clean_shed1["power_cuts_min"].mean())

Original Shed 1 mean: 3534.5089285714284
Clean Shed 1 mean: 58.67898383371824


In [6]:
# Check suspicious sensor rows in cleaned dataset

clean_stuck = clean[
    (clean["date"] >= "2026-03-01") &
    (clean["date"] <= "2026-03-20") &
    (clean["shift_hours"] == 10.8)
]

print("Stuck sensor rows in original:", 101)

print("Stuck sensor rows in cleaned:",
      len(clean_stuck))

print("\nShift hours distribution:")
print(clean["shift_hours"].value_counts().head(10))

Stuck sensor rows in original: 101
Stuck sensor rows in cleaned: 0

Shift hours distribution:
shift_hours
7.4     113
7.7      22
8.2      21
10.6     21
8.4      21
11.5     20
6.9      20
7.6      19
9.0      19
9.8      19
Name: count, dtype: int64


In [7]:
# Compare matching Shed 1 records using record_id

comparison = raw.merge(
    clean,
    on="record_id",
    suffixes=("_raw", "_clean")
)

shed1 = comparison[
    comparison["shed_raw"] == "Shed 1"
]

print(shed1[[
    "record_id",
    "power_cuts_min_raw",
    "power_cuts_min_clean"
]].head(10))

   record_id  power_cuts_min_raw  power_cuts_min_clean
1     R00002              5640.0                  94.0
5     R00006               300.0                   5.0
6     R00007              4980.0                  83.0
7     R00008              4140.0                  69.0
8     R00009              5940.0                  99.0
9     R00010              1440.0                  24.0
10    R00011              1140.0                  19.0
12    R00013              3600.0                  60.0
13    R00014              5880.0                  98.0
15    R00016              6360.0                 106.0


In [8]:
# Compare original and cleaned shift_hours
check = raw.merge(
    clean,
    on="record_id",
    suffixes=("_raw", "_clean")
)

affected = check[
    (check["date_raw"] >= "2026-03-01") &
    (check["date_raw"] <= "2026-03-20") &
    (check["shift_hours_raw"] == 10.8)
]

print("Original affected records:", len(affected))

print("\nBefore and after shift_hours:")
print(affected[[
    "record_id",
    "shift_hours_raw",
    "shift_hours_clean"
]].to_string(index=False))

Original affected records: 101

Before and after shift_hours:
record_id  shift_hours_raw  shift_hours_clean
   R00343             10.8                7.4
   R00343             10.8                7.4
   R00344             10.8                7.4
   R00345             10.8                7.4
   R00346             10.8                7.4
   R00347             10.8                7.4
   R00348             10.8                7.4
   R00349             10.8                7.4
   R00350             10.8                7.4
   R00350             10.8                7.4
   R00351             10.8                7.4
   R00352             10.8                7.4
   R00353             10.8                7.4
   R00354             10.8                7.4
   R00355             10.8                7.4
   R00356             10.8                7.4
   R00357             10.8                7.4
   R00358             10.8                7.4
   R00359             10.8                7.4
   R00360         

# Member 2 – MLflow Experiment Tracking

## 1. Dataset Preparation
Loading cleaned dataset and identifying features and target variable.

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/train_clean.csv")

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (1000, 11)


,record_id,date,loom_age_yr,thread_count,power_cuts_min,weaver_exp_yr,humidity_pct,shift_hours,fabric,shed,metres_per_shift
0,R00001,2026-01-01,18.0,132,40.0,1,89,10.8,silk,Shed 3,53.1
1,R00002,2026-01-01,29.0,64,94.0,8,78,7.6,cotton,Shed 1,27.8
2,R00003,2026-01-01,8.0,82,61.0,16,72,7.7,polyester,Shed 2,58.5
3,R00004,2026-01-01,19.0,103,39.0,13,81,8.2,cotton,Shed 2,50.2
4,R00005,2026-01-01,25.0,171,21.0,13,62,7.0,cotton,Shed 3,33.4


In [10]:
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Statistics:")
print(df["metres_per_shift"].describe())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   record_id         1000 non-null   str    
 1   date              1000 non-null   str    
 2   loom_age_yr       1000 non-null   float64
 3   thread_count      1000 non-null   int64  
 4   power_cuts_min    1000 non-null   float64
 5   weaver_exp_yr     1000 non-null   int64  
 6   humidity_pct      1000 non-null   int64  
 7   shift_hours       1000 non-null   float64
 8   fabric            1000 non-null   str    
 9   shed              1000 non-null   str    
 10  metres_per_shift  1000 non-null   float64
dtypes: float64(4), int64(3), str(4)
memory usage: 113.7 KB
None

Missing Values:
record_id           0
date                0
loom_age_yr         0
thread_count        0
power_cuts_min      0
weaver_exp_yr       0
humidity_pct        0
shift_hours         0
fabric              0
shed          

In [11]:
target = "metres_per_shift"

X = df.drop(columns=[target, "record_id", "date"])
y = df[target]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

display(X.head())

Features Shape: (1000, 8)
Target Shape: (1000,)


,loom_age_yr,thread_count,power_cuts_min,weaver_exp_yr,humidity_pct,shift_hours,fabric,shed
0,18.0,132,40.0,1,89,10.8,silk,Shed 3
1,29.0,64,94.0,8,78,7.6,cotton,Shed 1
2,8.0,82,61.0,16,72,7.7,polyester,Shed 2
3,19.0,103,39.0,13,81,8.2,cotton,Shed 2
4,25.0,171,21.0,13,62,7.0,cotton,Shed 3


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Separate numerical and categorical columns
numeric_features = [
    "loom_age_yr",
    "thread_count",
    "power_cuts_min",
    "weaver_exp_yr",
    "humidity_pct",
    "shift_hours"
]

categorical_features = ["fabric", "shed"]

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Preprocessing setup completed!")

Preprocessing setup completed!


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 800
Testing samples: 200



# 2. MLflow Experiment Tracking

## 2.1 MLflow Setup

MLflow is used to track machine learning experiments, log model parameters, evaluation metrics, and manage trained models.

### Objectives
- Configure MLflow tracking.
- Create a Power Loom Output Prediction experiment.
- Track model training results.
- Compare different model configurations.



## 2.2 Experiment 1 – Random Forest Regressor

**Model:** Random Forest Regressor

**Configuration:**
- Number of estimators: 100
- Maximum depth: 8
- Random state: 42

**Evaluation Metrics:**
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R-squared (R²)


In [14]:
import mlflow

# Set local MLflow tracking database
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Create or select experiment
mlflow.set_experiment("Power_Loom_Output_Prediction")

print("MLflow Experiment Created Successfully!")


2026/11/22 12:05:30 INFO mlflow.tracking.fluent: Experiment with name 'Power_Loom_Output_Prediction' does not exist. Creating a new experiment.


MLflow Experiment Created Successfully!


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Create model pipeline
rf_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=8,
        random_state=42
    ))
])

# Start MLflow run
with mlflow.start_run(run_name="Random_Forest_Experiment_1"):

    rf_pipeline.fit(X_train, y_train)

    predictions = rf_pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    # Log parameters
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 8)

    # Log metrics
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)

    print("Random Forest Experiment Completed!")
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)

Random Forest Experiment Completed!
MAE: 3.5815899943200953
RMSE: 4.335327118830609
R2: 0.8594681374787807


Experiment 2 – Random Forest

In [16]:
rf_pipeline_2 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        random_state=42
    ))
])

with mlflow.start_run(run_name="Random_Forest_Experiment_2"):

    rf_pipeline_2.fit(X_train, y_train)

    pred2 = rf_pipeline_2.predict(X_test)

    mae2 = mean_absolute_error(y_test, pred2)
    rmse2 = np.sqrt(mean_squared_error(y_test, pred2))
    r2_2 = r2_score(y_test, pred2)

    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 12)

    mlflow.log_metric("MAE", mae2)
    mlflow.log_metric("RMSE", rmse2)
    mlflow.log_metric("R2", r2_2)

    print("Experiment 2 completed!")
    print("MAE:", mae2)
    print("RMSE:", rmse2)
    print("R2:", r2_2)

Experiment 2 completed!
MAE: 3.505776065523433
RMSE: 4.233203242740571
R2: 0.8660109531810631


## Experiment 3 – Gradient Boosting

In [17]:
from sklearn.ensemble import GradientBoostingRegressor

gb_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ))
])

with mlflow.start_run(run_name="Gradient_Boosting_Experiment_3"):

    gb_pipeline.fit(X_train, y_train)

    pred3 = gb_pipeline.predict(X_test)

    mae3 = mean_absolute_error(y_test, pred3)
    rmse3 = np.sqrt(mean_squared_error(y_test, pred3))
    r2_3 = r2_score(y_test, pred3)

    mlflow.log_param("model", "GradientBoosting")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 3)
    mlflow.log_param("learning_rate", 0.1)

    mlflow.log_metric("MAE", mae3)
    mlflow.log_metric("RMSE", rmse3)
    mlflow.log_metric("R2", r2_3)

    print("Experiment 3 completed!")
    print("MAE:", mae3)
    print("RMSE:", rmse3)
    print("R2:", r2_3)

Experiment 3 completed!
MAE: 2.750647003669668
RMSE: 3.396937593804342
R2: 0.9137207832049499


In [18]:
experiment = mlflow.get_experiment_by_name(
    "Power_Loom_Output_Prediction"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

results = runs[
    ["tags.mlflow.runName", "metrics.MAE",
     "metrics.RMSE", "metrics.R2"]
].sort_values("metrics.MAE")

display(results)

,tags.mlflow.runName,metrics.MAE,metrics.RMSE,metrics.R2
0,Gradient_Boosting_Experiment_3,2.750647,3.396938,0.913721
1,Random_Forest_Experiment_2,3.505776,4.233203,0.866011
2,Random_Forest_Experiment_1,3.581590,4.335327,0.859468


In [19]:
best_run = runs.loc[runs["metrics.MAE"].idxmin()]

print("Best Run ID:", best_run["run_id"])
print("Best Model MAE:", best_run["metrics.MAE"])
print("Best Model:", best_run["tags.mlflow.runName"])

Best Run ID: 4fad491dd9124585b9966d6f4b05d250
Best Model MAE: 2.750647003669668
Best Model: Gradient_Boosting_Experiment_3


## Model Registration and Champion Alias

In [22]:
# Save best Gradient Boosting model using cloudpickle

with mlflow.start_run(run_id=best_run["run_id"]):

    mlflow.sklearn.log_model(
        sk_model=gb_pipeline,
        name="model",
        serialization_format="cloudpickle"
    )

print("Best model artifact saved successfully!")

2026/11/22 12:17:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Best model artifact saved successfully!


In [23]:
model_name = "PowerLoom_Output_Model"

model_uri = f"runs:/{best_run['run_id']}/model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print("Model Registered Successfully!")
print("Model Name:", registered_model.name)
print("Model Version:", registered_model.version)

Registered model 'PowerLoom_Output_Model' already exists. Creating a new version of this model...
2026/11/22 12:18:42 WARNING mlflow.tracking._model_registry.fluent: Run with id 4fad491dd9124585b9966d6f4b05d250 has no artifacts at artifact path 'model', registering model based on models:/m-1910ace9a7244a57b7a56add8ee9237e instead


Model Registered Successfully!
Model Name: PowerLoom_Output_Model
Model Version: 1


Created version '1' of model 'PowerLoom_Output_Model'.


In [24]:
from mlflow import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=registered_model.version
)

print("Champion alias assigned successfully!")

Champion alias assigned successfully!


## Experiment Comparison and Best Model Selection

In [25]:
champion = client.get_model_version_by_alias(
    name="PowerLoom_Output_Model",
    alias="champion"
)

print("Model Name:", champion.name)
print("Version:", champion.version)
print("Champion Alias:", "champion")
print("Run ID:", champion.run_id)

Model Name: PowerLoom_Output_Model
Version: 1
Champion Alias: champion
Run ID: 4fad491dd9124585b9966d6f4b05d250
